# 01 — Data inspection and cleaning
PRD §7. Every cleaning rule below is justified by a check on the raw file.

In [1]:
import sys, json
sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config, cleaning, features, baseline
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid')
config.FIG_DIR.mkdir(parents=True, exist_ok=True)
def save(name): plt.tight_layout(); plt.savefig(config.FIG_DIR / f'{name}.png', dpi=130)

In [2]:
raw = cleaning.load_raw()
raw.head()

,user_id,session_id,event_index,timestamp,key_event,dwell_time_ms,flight_time_ms,is_correction,valence,arousal,mood_state
0,U001,1,1,2026-01-01 09:00:00.107581429,backspace,98.077428,107.581429,0,0.355528,0.193335,calm
1,U001,1,2,2026-01-01 09:00:00.175848514,backspace,94.919692,68.267085,1,-0.244586,0.774009,stressed
2,U001,1,3,2026-01-01 09:00:00.265080330,l,117.265447,89.231816,0,-0.286123,0.823853,stressed
3,U001,1,4,2026-01-01 09:00:00.369037867,w,172.366542,103.957537,0,0.323918,0.235477,calm
4,U001,1,5,2026-01-01 09:00:00.524382406,p,75.121530,155.344539,0,0.109457,0.165179,calm


## Audit of the raw file

In [3]:
report = cleaning.audit(raw)
for k, v in report.items():
    print(f'{k:35s} {v}')

rows                                100000
users                               50
sessions_per_user                   20.0
events_per_session                  {'count': 1000.0, 'mean': 100.0, 'std': 0.0, 'min': 100.0, '25%': 100.0, '50%': 100.0, '75%': 100.0, 'max': 100.0}
duplicate_rows                      0
missing                             {'dwell_time_ms': 150, 'flight_time_ms': 150}
negative_dwell                      100
negative_flight                     0
dwell_over_max                      0
flight_over_long_pause              695
unparseable_timestamps              0
sessions_not_time_ordered           0
session_duration_s                  {'count': 1000.0, 'mean': 13.880472081358999, 'std': 3.040292949466696, 'min': 8.064280592, '25%': 11.69147020025, '50%': 13.381371025, '75%': 15.70096600275, 'max': 25.588258562}
raw_key_values                      30
raw_keys_are_characters             True
backspace_not_flagged_correction    2348
correction_not_backspace            0

### Privacy check: `key_event` stores real characters
R02 requires that typed content is never stored. The raw file contains the actual letters, so they are mapped to categories during cleaning and never written back out.

In [4]:
raw['key_event'].value_counts().head(10)

key_event
space        14559
backspace     6294
r             3077
m             3037
q             3016
o             3008
y             3007
h             3001
i             2997
t             2989
Name: count, dtype: int64

### Label structure: mood changes almost every keystroke
This is the most important property of the dataset and limits everything downstream (see notebook 03).

In [5]:
raw.loc[raw.user_id == 'U001'].head(15)[['session_id', 'event_index', 'key_event', 'mood_state', 'valence', 'arousal']]

,session_id,event_index,key_event,mood_state,valence,arousal
0,1,1,backspace,calm,0.355528,0.193335
1,1,2,backspace,stressed,-0.244586,0.774009
2,1,3,l,stressed,-0.286123,0.823853
3,1,4,w,calm,0.323918,0.235477
4,1,5,p,calm,0.109457,0.165179
5,1,6,space,low,-0.468530,-0.341891
6,1,7,v,stressed,-0.383017,0.959393
7,1,8,p,stressed,-0.264653,0.875202
8,1,9,o,calm,0.139666,0.159659
9,1,10,v,calm,0.304813,0.020569


### Invalid and missing timing values

In [6]:
print(raw[['dwell_time_ms', 'flight_time_ms']].describe().round(1))
print('negative dwell:', (raw.dwell_time_ms < 0).sum())

       dwell_time_ms  flight_time_ms
count        99850.0         99850.0
mean           108.5           142.7
std             34.0           230.4
min            -49.6            20.0
25%             84.6            81.5
50%            103.7           109.5
75%            127.4           148.4
max            394.9          3468.8
negative dwell: 100


`flight_time_ms` is the difference between consecutive timestamps, so a missing flight can be recovered exactly instead of imputed:

In [7]:
ts = pd.to_datetime(raw.timestamp)
gap = ts.groupby([raw.user_id, raw.session_id]).diff().dt.total_seconds() * 1000
print('max |gap - flight| (ms):', (gap - raw.flight_time_ms).abs().quantile([.5, .99]).round(6).to_dict())

max |gap - flight| (ms): {0.5: 0.0, 0.99: 1e-06}


### Correction flag
Every `is_correction=1` row is a backspace, but some backspaces are not flagged. A backspace is a correction by definition, so the flag is recomputed from the key category.

In [8]:
pd.crosstab(raw.key_event.eq('backspace'), raw.is_correction, rownames=['is backspace'])

is_correction,0,1
is backspace,,
False,93706,0
True,2348,3946


## Apply cleaning

In [9]:
clean, log = cleaning.clean(raw)
print('\n'.join(log))
clean.to_csv(config.CLEAN_PATH, index=False)
clean.head()

Removed 0 exact duplicate rows.
Dropped 0 rows with unparseable timestamps.
Sorted by user/session/time and re-indexed events per session (raw event_index does not restart at each session).
Replaced key_event with key_category {char, space, backspace, enter, shift}; the raw characters are not carried forward.
Set is_correction=1 for 2348 backspace events that were not flagged.
Marked 100 invalid dwell values (<0 or >1000 ms) and 0 negative flight values as missing.
Recovered 150 missing flight values from timestamp differences.
Imputed 250 remaining missing dwell_time_ms values with the user's median.
Imputed 0 remaining missing flight_time_ms values with the user's median.
Kept long flights as pauses (not outliers): 1073 events > 500 ms form a separate mode in the flight distribution.


,user_id,session_id,timestamp,dwell_time_ms,flight_time_ms,is_correction,valence,arousal,mood_state,event_in_session,key_category,flight_imputed,dwell_imputed,is_pause
0,U001,1,2026-01-01 09:00:00.107581429,98.077428,107.581429,1,0.355528,0.193335,calm,1,backspace,False,False,False
1,U001,1,2026-01-01 09:00:00.175848514,94.919692,68.267085,1,-0.244586,0.774009,stressed,2,backspace,False,False,False
2,U001,1,2026-01-01 09:00:00.265080330,117.265447,89.231816,0,-0.286123,0.823853,stressed,3,char,False,False,False
3,U001,1,2026-01-01 09:00:00.369037867,172.366542,103.957537,0,0.323918,0.235477,calm,4,char,False,False,False
4,U001,1,2026-01-01 09:00:00.524382406,75.121530,155.344539,0,0.109457,0.165179,calm,5,char,False,False,False


In [10]:
assert 'key_event' not in clean.columns and set(clean.key_category) <= {'char','space','backspace','enter','shift'}
assert clean[['dwell_time_ms','flight_time_ms']].isna().sum().sum() == 0 and (clean.dwell_time_ms > 0).all()
print(clean.shape, clean.groupby('user_id').size().describe()[['min','max']].to_dict())

(100000, 14) {'min': 2000.0, 'max': 2000.0}


## Summary of cleaning decisions
| Problem found | Decision | Why |
|---|---|---|
| Real characters in `key_event` | Map to 5 categories, drop original | R02: typed content must not be stored |
| 2,348 unflagged backspaces | `is_correction = (key == backspace)` | Every flagged correction is a backspace |
| 100 negative dwell times | Set missing, impute with user median | A key cannot be held for negative time |
| 150 missing flight times | Recover from timestamp differences, else user median | Flight equals the timestamp gap to < 1 µs |
| 150 missing dwell times | User median | Users differ in dwell (between-user CV ≈ 8%), so a global median would bias them |
| Flights > 500 ms (~1%) | **Kept** as pauses | They form a separate mode (see 02_eda plot 2); they are behaviour, not noise |
| No duplicates, unparseable timestamps or out-of-order sessions | Nothing to do | Checked anyway |